[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/02_Vision_Language_Models/03_visual_question_answering/03_visual_question_answering.ipynb)

# 03. Visual Question Answering (VQA)

**This notebook covers:**
- VQA architecture: Image + Question → Answer
- Building a VQA model from scratch
- Visualizing which image regions answer which questions
- Comparison of VQA approaches

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/02_Vision_Language_Models/03_visual_question_answering")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

In [ ]:
# Visualize VQA task
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('Visual Question Answering (VQA) Pipeline', fontsize=18, fontweight='bold', pad=20)

draw_architecture_block(ax, 2, 6, 3, 0.8, 'Image', '#E74C3C')
draw_architecture_block(ax, 7, 6, 4, 0.8, 'Question: "What color\nis the cat?"', '#3498DB', fontsize=9)

draw_architecture_block(ax, 2, 4.3, 3, 0.8, 'Vision Encoder', '#C0392B')
draw_architecture_block(ax, 7, 4.3, 4, 0.8, 'Text Encoder', '#2980B9')

draw_architecture_block(ax, 5, 2.5, 8, 1, 'Cross-Modal Fusion\n(Cross-Attention or Concatenation)', '#9B59B6')

draw_architecture_block(ax, 5, 0.8, 4, 0.8, 'Answer: "orange"', '#2ECC71')

draw_arrow(ax, (2, 5.5), (2, 4.8))
draw_arrow(ax, (7, 5.5), (7, 4.8))
draw_arrow(ax, (2, 3.8), (3, 3.1))
draw_arrow(ax, (7, 3.8), (7, 3.1))
draw_arrow(ax, (5, 1.9), (5, 1.3))

# Types of VQA
ax.text(11.5, 3.5, 'VQA Types:', fontsize=11, fontweight='bold')
vqa_types = ['Classification\n(pick from answers)', 'Generation\n(generate answer)', 'Yes/No\n(binary)']
for i, t in enumerate(vqa_types):
    ax.text(11.5, 2.7 - i*0.8, f'• {t}', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/vqa_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

### Cross-Attention Fusion Math

The core fusion mechanism in VQA lets **question tokens query image patches** to locate relevant visual evidence.

Given question token embeddings $T \in \mathbb{R}^{T \times d}$ and image patch embeddings $I \in \mathbb{R}^{N \times d}$:

$$Q = T W^Q, \quad K = I W^K, \quad V = I W^V$$

where $W^Q, W^K, W^V \in \mathbb{R}^{d \times d_k}$ are learned projection matrices.

$$\text{Attn}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

**Shape intuition** (batch size $B$):

| Tensor | Shape | Role |
|--------|-------|------|
| $Q$ | $[B, T, d_k]$ | Each question word asks "where in the image?" |
| $K, V$ | $[B, N, d_k]$ | Image patches provide keys and values |
| Attention weights | $[B, T, N]$ | Word $t$'s focus on patch $n$ |
| Output | $[B, T, d_k]$ | Image-grounded question representation |

Each question word independently attends over all image patches. For "What **color** is the **cat**?", the word "color" should attend to the cat's body patches, while "cat" attends to the cat's silhouette — this is exactly what the attention visualization below demonstrates.

### Pooling Strategies

After cross-attention fusion, we have a sequence of token representations $h_1, \ldots, h_T \in \mathbb{R}^d$. We must collapse this sequence into a single vector $h \in \mathbb{R}^d$ for the answer classifier.

#### Mean Pooling

$$h = \frac{1}{T}\sum_{t=1}^{T} h_t$$

Treats all question tokens equally. Simple and robust — works well when every word contributes (e.g., "What color is the cat?" — all words matter).

#### [CLS] Pooling

$$h = h_0$$

Uses only the first (classification) token, which has attended to all other tokens via self-attention. Relies on the encoder to aggregate information into a single vector — same idea as BERT.

#### Attention Pooling (learned)

$$h = \sum_{t=1}^{T} \alpha_t \, h_t, \qquad \alpha = \text{softmax}(w^\top h_t)$$

A learned weight vector $w$ assigns **importance scores** to each token. The model can learn to focus on content words ("color", "cat") and down-weight function words ("is", "the"). Most flexible — adds only $d$ parameters for $w$.

In [ ]:
# Build a VQA model from scratch

class VQAModel(nn.Module):
    """Simple VQA: Image + Question → Answer (classification)."""
    def __init__(self, vocab_size=3000, embed_dim=128, n_heads=4,
                 n_answers=100, img_size=32, patch_size=4):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2

        # Image encoder
        self.patch_embed = nn.Conv2d(3, embed_dim, patch_size, patch_size)
        self.img_pos = nn.Parameter(torch.randn(1, n_patches, embed_dim) * 0.02)
        img_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim*4, batch_first=True
        )
        self.img_encoder = nn.TransformerEncoder(img_layer, num_layers=2)

        # Question encoder
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.q_pos = nn.Embedding(64, embed_dim)
        q_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim*4, batch_first=True
        )
        self.q_encoder = nn.TransformerEncoder(q_layer, num_layers=2)

        # Cross-attention: question attends to image
        self.cross_attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

        # Answer classifier
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim * 2, n_answers)
        )

    def forward(self, images, question_ids, return_attention=False):
        # Encode image
        img_feat = self.patch_embed(images).flatten(2).transpose(1, 2)
        img_feat = img_feat + self.img_pos
        img_feat = self.img_encoder(img_feat)  # [B, N_patches, D]

        # Encode question
        B, T = question_ids.shape
        pos = torch.arange(T, device=question_ids.device).unsqueeze(0).expand(B, -1)
        q_feat = self.token_embed(question_ids) + self.q_pos(pos)
        q_feat = self.q_encoder(q_feat)  # [B, T, D]

        # Cross-attention: Q=question, K/V=image
        cross_out, attn_weights = self.cross_attn(q_feat, img_feat, img_feat)
        fused = self.norm(q_feat + cross_out)

        # Pool and classify
        pooled = fused.mean(dim=1)
        logits = self.classifier(pooled)

        if return_attention:
            return logits, attn_weights
        return logits


model = VQAModel(vocab_size=3000, embed_dim=128, n_answers=100)
count_parameters(model)

# Test
imgs = torch.randn(2, 3, 32, 32)
q_ids = torch.randint(0, 3000, (2, 12))
logits, attn = model(imgs, q_ids, return_attention=True)
print(f"\nOutput: {logits.shape}")
print(f"Cross-attention: {attn.shape} (question_tokens × image_patches)")

### VQA Loss Functions

How we define the training target depends on the dataset annotation scheme.

#### Hard Labels (standard classification)

When each question has a single ground-truth answer $a^*$:

$$\mathcal{L} = -\log P(a^* \mid I, Q)$$

Standard cross-entropy over the answer vocabulary. Used when annotations are unambiguous.

#### Soft Labels (VQA v2)

VQA v2 collects **10 annotator answers** per question. Different people may disagree:

> Q: "What color is the sky?" → "blue" (8 annotators), "light blue" (2 annotators)

Instead of picking one answer, distribute target mass proportionally:

$$s_a = \min\left(1,\; \frac{\text{count}_a}{3}\right)$$

$$\mathcal{L} = -\sum_a s_a \log P(a \mid I, Q)$$

The $\min(\cdot, 1)$ cap prevents a single dominant answer from saturating the target. Soft labels **reduce overfitting** to annotation noise and reward the model for capturing answer diversity.

#### Why soft labels matter

| Annotation | Hard label | Soft label effect |
|------------|-----------|-------------------|
| "blue" × 8, "light blue" × 2 | Force "blue" only | Model learns both are valid |
| "2" × 6, "3" × 4 | Force "2" | Partial credit for "3" |
| All 10 agree | Same as hard label | $s_a = 1$ for correct answer |

This is why VQA v2 accuracy is reported as **min(1, #humans/3)** agreement — the metric mirrors the training objective.

In [ ]:
# Visualize cross-attention: which patches does each question word attend to?

question_words = ['what', 'color', 'is', 'the', 'cat', '?', '[PAD]'] + ['[PAD]']*5
attn_map = attn[0].detach().numpy()  # [Q_len, N_patches]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('VQA Cross-Attention: Which image patches does each word attend to?',
             fontsize=14, fontweight='bold')

patch_grid = 8  # sqrt(64)
for idx, ax in enumerate(axes.flat):
    if idx >= len(question_words) or question_words[idx] == '[PAD]':
        ax.axis('off')
        continue
    
    word_attn = attn_map[idx].reshape(patch_grid, patch_grid)
    ax.imshow(word_attn, cmap='hot', interpolation='nearest')
    ax.set_title(f'Word: "{question_words[idx]}"', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('../assets/vqa_attention.png', dpi=150, bbox_inches='tight')
plt.show()
print("After training, 'cat' should attend to cat patches, 'color' to those patches too!")

### VQA Question Types

VQA benchmarks span a wide range of reasoning difficulty. Understanding question type helps choose architecture and evaluation strategy.

| Type | Example | Approach | Difficulty |
|------|---------|----------|------------|
| **Yes/No** | "Is there a cat?" | Binary classification (2 classes) | Easy |
| **Counting** | "How many dogs?" | Regression or classification over numbers | Medium |
| **Color** | "What color is the car?" | Classification over color vocabulary | Medium |
| **Spatial** | "What is to the left of the table?" | Requires spatial reasoning + object detection | Hard |
| **Why/How** | "Why is the person smiling?" | Requires commonsense / world knowledge | Very Hard |

**Distribution in VQA v2:** ~38% yes/no, ~12% counting, ~15% color, remainder open-ended. Models often excel at yes/no but struggle with spatial and commonsense questions — cross-attention alone is insufficient for "why" questions without external knowledge.

### VQA Approaches Comparison

| Approach | Architecture | Strengths | Weaknesses |
|----------|-------------|-----------|------------|
| **Classification** | ViT + BERT + MLP | Fast inference; simple training; works well for closed-vocabulary VQA (VQA v2 has ~3K answer classes) | Fixed answer set — cannot handle novel answers; poor generalization to open-ended questions |
| **Generative** | Encoder-Decoder (BLIP, BLIP-2) | Open-ended answers; can produce multi-word responses; leverages pretrained vision-language models | Slower decoding; harder to train; evaluation is trickier (exact string match vs. semantic equivalence) |
| **LLM-based (LLaVA)** | ViT + projection layer + LLM (LLaMA/Vicuna) | Best open-ended quality; strong commonsense and reasoning; instruction-following | Heavy compute (~7B+ params); requires LoRA/QLoRA for fine-tuning; latency prohibitive for real-time apps |

**When to choose which:**
- **Classification** → benchmark datasets with fixed answer vocab (VQA v2, GQA)
- **Generative** → need flexible phrasing without full LLM cost
- **LLM-based** → research, chatbots, or tasks requiring reasoning ("Why is the person smiling?")

## VQA Approach Comparison

| Approach | Model | Params | Low Compute? |
|----------|-------|--------|--------|
| Classification | ViT + BERT + MLP | ~50M | Yes |
| Generative | BLIP / BLIP-2 | ~200M-3B | Base model OK |
| LLM-based | LLaVA | ~7B+ | Need LoRA/QLoRA |

---
**Next:** Module 03 - Training Strategies (the core of what you want to learn!)